In [27]:
# Load datasets
import pandas as pd

train_data = pd.read_csv("../data/processed/train_data.csv")
validation_data = pd.read_csv("../data/processed/validation_data.csv")
test_data = pd.read_csv("../data/processed/test_data.csv")


In [28]:
train_data.dtypes

YEAR                            float64
FIRE_NUMBER                      object
SIZE_CLASS                       object
FIRE_ORIGIN                      object
DETECTION_AGENT_TYPE             object
REPORTED_DATE                    object
DISPATCHED_RESOURCE              object
DISPATCH_DATE                    object
ASSESSMENT_DATETIME              object
ASSESSMENT_HECTARES             float64
FIRE_SPREAD_RATE                float64
FIRE_TYPE                        object
FIRE_POSITION_ON_SLOPE           object
WEATHER_CONDITIONS_OVER_FIRE     object
TEMPERATURE                     float64
RELATIVE_HUMIDITY               float64
WIND_DIRECTION                   object
WIND_SPEED                      float64
FUEL_TYPE                        object
FOREST_AREA                      object
dtype: object

#### Select attributes for prediction

In [29]:
# Exclude target, identifiers, and raw timestamps
exclude_columns = [
    "SIZE_CLASS",
    "FIRE_NUMBER",
    "REPORTED_DATE",
    "DISPATCH_DATE",
    "ASSESSMENT_DATETIME"
]

feature_columns = [
    col for col in train_data.columns
    if col not in exclude_columns
]

X_train = train_data[feature_columns].copy()
y_train = train_data["SIZE_CLASS"]

X_validation = validation_data[feature_columns].copy()
y_validation = validation_data["SIZE_CLASS"]

X_test = test_data[feature_columns].copy()
y_test = test_data["SIZE_CLASS"]



#### convert categorical columns to numerical

In [32]:

# indentify numerical and categorical columns
numerical_columns = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Encode categorical columns using one-hot encoding

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=int
)

X_validation = pd.get_dummies(
    X_validation,
    columns=categorical_columns,
    dtype=int
).reindex(columns=X_train.columns, fill_value=0)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=int
).reindex(columns=X_train.columns, fill_value=0)

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

X_train.dtypes


Training: (19479, 82)
Validation: (4174, 82)
Testing: (4175, 82)


YEAR                   float64
ASSESSMENT_HECTARES    float64
FIRE_SPREAD_RATE       float64
TEMPERATURE            float64
RELATIVE_HUMIDITY      float64
                        ...   
FOREST_AREA_M            int64
FOREST_AREA_P            int64
FOREST_AREA_R            int64
FOREST_AREA_S            int64
FOREST_AREA_W            int64
Length: 82, dtype: object

In [31]:
print("Forest area included:", 
      any("FOREST_AREA" in col for col in X_train.columns))

print("Missing values:", X_train.isnull().sum().sum())

print("Non-numeric columns:",
      X_train.select_dtypes(exclude="number").columns.tolist())

Forest area included: True
Missing values: 0
Non-numeric columns: []
